In [1]:
import torch
import torch.nn as nn

import time

In [2]:
mat = nn.Linear(1024, 1024, bias=False).to("cuda")

def mat_mul():

    for i in range(1000):
        _ = mat(torch.randn(1024, 1024, device="cuda"))

In [3]:
# without autocast

# Warm-up
with torch.no_grad():
    mat_mul()
torch.cuda.synchronize()

for i in range(10):
    t0 = time.time()
    with torch.no_grad():
        mat_mul()
    torch.cuda.synchronize()
    t1 = time.time()
    print(f"Step {i}: {(t1 - t0)*1000:.4f} ms")


Step 0: 237.3931 ms
Step 1: 237.5376 ms
Step 2: 242.7475 ms
Step 3: 244.2198 ms
Step 4: 238.4858 ms
Step 5: 237.8454 ms
Step 6: 238.0760 ms
Step 7: 243.3906 ms
Step 8: 239.0234 ms
Step 9: 237.8106 ms


In [4]:
# with TF32 instead of FP32 (same memory usage but faster computations)

torch.set_float32_matmul_precision('high')  # Enable TF32 for matmul operations

# Warm-up
with torch.no_grad():
    mat_mul()
torch.cuda.synchronize()

for i in range(10):
    t0 = time.time()
    with torch.no_grad():
        mat_mul()
    torch.cuda.synchronize()
    t1 = time.time()
    print(f"Step {i}: {(t1 - t0)*1000:.4f} ms")
    
torch.set_float32_matmul_precision('highest')  # Reset to default precision FP32 matmul operations

Step 0: 123.2362 ms
Step 1: 123.2326 ms


Step 2: 123.2388 ms
Step 3: 123.5244 ms
Step 4: 123.1389 ms
Step 5: 123.3528 ms
Step 6: 123.7044 ms
Step 7: 124.1856 ms
Step 8: 123.8317 ms
Step 9: 123.9965 ms


In [5]:
# with bfloat16 autocast

# Warm-up
with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        mat_mul()
torch.cuda.synchronize()

for i in range(10):
    t0 = time.time()
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            mat_mul()
    torch.cuda.synchronize()
    t1 = time.time()
    print(f"Step {i}: {(t1 - t0)*1000:.4f} ms")

Step 0: 77.7769 ms
Step 1: 65.2041 ms
Step 2: 67.0738 ms
Step 3: 80.6940 ms
Step 4: 76.2846 ms
Step 5: 66.7455 ms
Step 6: 68.2304 ms
Step 7: 82.5839 ms
Step 8: 87.4319 ms
Step 9: 93.1177 ms


In [6]:
# with float16 autocast (not recommended) may required gradient scaling for training
#  float16 has a much smaller dynamic range than bfloat16, making it prone to overflow (→ inf) and underflow (→ 0) during training. Gradient scaling (via GradScaler) compensates — but for inference (which is what this benchmark does), both are generally fine.

# Warm-up
with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        mat_mul()
torch.cuda.synchronize()

for i in range(10):
    t0 = time.time()
    
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            mat_mul()
    torch.cuda.synchronize()
    
    t1 = time.time()
    print(f"Step {i}: {(t1 - t0)*1000:.4f} ms")

Step 0: 68.8992 ms
Step 1: 96.1924 ms
Step 2: 82.1481 ms
Step 3: 71.1913 ms
Step 4: 88.6717 ms
Step 5: 68.3668 ms
Step 6: 77.2192 ms
Step 7: 87.2011 ms
Step 8: 69.2837 ms
Step 9: 88.0899 ms
